# Day 3 - Baseline Training and Analysis

This notebook is a single place to review what was trained, compare runs, inspect metrics/confusion matrices, and visualize prediction failures (especially false negatives).

## Scope
- Dataset size and class balance
- Baseline run comparisons
- Training dynamics (loss/recall/gap)
- Confusion matrix and FN/FP counts
- Misclassification and false-negative image review
- Sample predictions from saved checkpoints


## Day 4 Concept - What This CNN Learns

### Early Layer (Conv16)
- Tire edges and sidewall boundaries
- Local contrast between rubber and grooves

### Mid Layer (Conv32)
- Groove geometry and pattern repetition
- Local texture regularity

### Final Decision Stage (Flatten + Linear + Sigmoid)
- Pattern disruptions and crack-like breaks
- Outputs final defect probability

### Feature Maps (Important)
- Each conv layer outputs multiple channels.
- Each channel is a different learned detector response.
- `Conv(16, ...)` gives **16 feature maps** (16 learned views of the same image).


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

# Resolve repo root first, then set writable cache dirs for notebook runtime.
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    for p in Path.cwd().parents:
        if (p / "src").exists():
            ROOT = p
            break

os.environ.setdefault("MPLCONFIGDIR", str(ROOT / "artifacts/.mpl"))
os.environ.setdefault("XDG_CACHE_HOME", str(ROOT / "artifacts/.cache"))
os.environ.setdefault("TORCH_HOME", str(ROOT / "artifacts/.torch"))
(ROOT / "artifacts/.mpl").mkdir(parents=True, exist_ok=True)
(ROOT / "artifacts/.cache").mkdir(parents=True, exist_ok=True)
(ROOT / "artifacts/.torch").mkdir(parents=True, exist_ok=True)

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import Markdown, display

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

if "torchvision" in sys.modules:
    _tv = sys.modules.get("torchvision")
    if _tv is not None and not hasattr(_tv, "extension"):
        for _k in [k for k in list(sys.modules) if k == "torchvision" or k.startswith("torchvision.")]:
            sys.modules.pop(_k, None)
        print("Cleared stale partial torchvision modules from kernel state.")


MANIFEST_PATH = ROOT / "data/processed/D1_manifest.csv"
RUN_BASE = ROOT / "artifacts/day3_baseline"

# Map display name -> run folder
RUNS = {
    "SimpleCNN v1 (1ep)": "simple_cnn_v1",
    "SimpleCNN v2 (2ep)": "simple_cnn_v2",
    "Frozen ResNet18 (2ep)": "frozen_resnet18_v1",
    "ResNet18 + unfrozen layer4 (2ep)": "frozen_resnet18_unfreeze_v2",
    "ResNet50 + unfrozen layer4 (1ep)": "frozen_resnet50_unfreeze_v1",
    "ResNet50 + unfrozen layer4 (3ep)": "frozen_resnet50_unfreeze_v2_3ep",
}

print(f"Repo root: {ROOT}")
print(f"Manifest: {MANIFEST_PATH}")
print(f"Run base: {RUN_BASE}")
print("\nAvailable run directories:")
if RUN_BASE.exists():
    for r in sorted(RUN_BASE.glob("*")):
        if r.is_dir():
            print(" -", r.name)
else:
    print("Run base directory not found yet.")

In [ ]:
def _read_json(path: Path):
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return None


def load_run_artifacts(run_dir: Path) -> dict:
    return {
        "run_dir": run_dir,
        "metrics": _read_json(run_dir / "metrics.json"),
        "eval": _read_json(run_dir / "eval_report.json"),
        "model_info": _read_json(run_dir / "model_info.json"),
        "misclassified_csv": run_dir / "misclassified.csv",
        "false_negatives_csv": run_dir / "false_negatives.csv",
        "misclassified_grid": run_dir / "misclassified_grid.png",
        "false_negative_grid": run_dir / "false_negative_grid.png",
    }


RUN_DATA = {}
for name, folder in RUNS.items():
    run_dir = RUN_BASE / folder
    if run_dir.exists():
        RUN_DATA[name] = load_run_artifacts(run_dir)

print(f"Loaded {len(RUN_DATA)} runs with directories found.")
for k in RUN_DATA:
    has_eval = RUN_DATA[k].get("eval") is not None
    print(f" - {k} | eval_report={'yes' if has_eval else 'no'}")

if not RUN_DATA:
    display(Markdown("**No run directories found yet. Train/evaluate at least one model first.**"))


## 1) Dataset size and class balance


In [ ]:
if not MANIFEST_PATH.exists():
    display(Markdown(f"**Manifest not found:** `{MANIFEST_PATH}`. Run `scripts/prepare_manifests.py` first."))
    manifest_df = pd.DataFrame(columns=["split", "label_str", "label", "image_path"])
else:
    manifest_df = pd.read_csv(MANIFEST_PATH)

total_images = len(manifest_df)
if total_images == 0:
    display(Markdown("**Manifest is empty.**"))
else:
    class_counts = manifest_df["label_str"].value_counts()
    split_counts = pd.crosstab(manifest_df["split"], manifest_df["label_str"])

    display(Markdown(f"**Total images:** {total_images}"))
    display(class_counts.to_frame("count"))
    display(split_counts)

    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    sns.barplot(x=class_counts.index, y=class_counts.values, ax=ax[0], palette="Set2")
    ax[0].set_title("Class Balance")
    for i, v in enumerate(class_counts.values):
        ax[0].text(i, v + 5, str(v), ha="center")

    split_counts.plot(kind="bar", stacked=True, ax=ax[1], colormap="viridis")
    ax[1].set_title("Split-wise Class Counts")
    ax[1].set_ylabel("Count")
    plt.tight_layout()
    plt.show()


## 2) Baseline run comparison (metrics + confusion)


In [ ]:
rows = []
for run_name, data in RUN_DATA.items():
    ev = data.get("eval")
    if not ev:
        continue
    m = ev.get("metrics", {})
    cm = m.get("confusion", [[None, None], [None, None]])
    fp = cm[0][1] if cm and cm[0][1] is not None else None
    fn = cm[1][0] if cm and cm[1][0] is not None else None
    rows.append({
        "run": run_name,
        "accuracy": m.get("accuracy"),
        "precision_macro": m.get("precision"),
        "recall_macro": m.get("recall"),
        "f1_macro": m.get("f1_macro"),
        "precision_defect": m.get("precision_defect"),
        "recall_defect": m.get("recall_defect"),
        "f1_defect": m.get("f1_defect"),
        "auroc": m.get("auroc"),
        "auprc": m.get("auprc"),
        "false_positives": fp,
        "false_negatives": fn,
    })

comparison_df = pd.DataFrame(rows)
if comparison_df.empty:
    display(Markdown("**No eval_report.json found in available runs yet.**"))
else:
    comparison_df = comparison_df.sort_values(["recall_defect", "f1_defect"], ascending=False).reset_index(drop=True)
    display(comparison_df)

    plot_cols = ["accuracy", "recall_defect", "f1_defect", "auroc"]
    melted = comparison_df[["run", *plot_cols]].melt(id_vars="run", var_name="metric", value_name="value")
    plt.figure(figsize=(12, 5))
    sns.barplot(data=melted, x="run", y="value", hue="metric")
    plt.xticks(rotation=25, ha="right")
    plt.ylim(0.7, 1.01)
    plt.title("Baseline Comparison")
    plt.tight_layout()
    plt.show()


## 3) Training dynamics (loss, recall, validation gap)


In [ ]:
preferred = "ResNet50 + unfrozen layer4 (3ep)"
RUN_TO_PLOT = preferred if preferred in RUN_DATA else (next(iter(RUN_DATA.keys())) if RUN_DATA else None)

if RUN_TO_PLOT is None:
    display(Markdown("**No runs available to plot training dynamics.**"))
else:
    hist = (RUN_DATA[RUN_TO_PLOT].get("metrics") or {}).get("history")
    if not hist:
        display(Markdown(f"**No history found for run:** `{RUN_TO_PLOT}`"))
    else:
        train_hist = pd.DataFrame(hist.get("train", []))
        val_hist = pd.DataFrame(hist.get("val", []))
        epochs = np.arange(1, len(train_hist) + 1)

        fig, ax = plt.subplots(1, 3, figsize=(15, 4))
        ax[0].plot(epochs, train_hist["loss"], marker="o", label="train_loss")
        ax[0].plot(epochs, val_hist["loss"], marker="o", label="val_loss")
        ax[0].set_title("Loss Curves")
        ax[0].set_xlabel("Epoch")
        ax[0].legend()

        ax[1].plot(epochs, val_hist["recall_defect"], marker="o", color="tab:red")
        ax[1].axhline(0.95, ls="--", c="gray", lw=1)
        ax[1].set_ylim(0.0, 1.01)
        ax[1].set_title("Val Recall (Defect)")
        ax[1].set_xlabel("Epoch")

        gap = train_hist["loss"].values - val_hist["loss"].values
        ax[2].plot(epochs, gap, marker="o", color="tab:green")
        ax[2].axhline(0.0, ls="--", c="gray", lw=1)
        ax[2].set_title("Train-Val Loss Gap")
        ax[2].set_xlabel("Epoch")

        plt.suptitle(RUN_TO_PLOT)
        plt.tight_layout()
        plt.show()

        summary = pd.DataFrame({
            "epoch": epochs,
            "train_loss": train_hist["loss"],
            "val_loss": val_hist["loss"],
            "val_recall_defect": val_hist["recall_defect"],
            "val_f1_defect": val_hist["f1_defect"],
            "loss_gap_train_minus_val": gap,
        })
        display(summary)


## 4) Confusion matrix view


In [ ]:
preferred = "ResNet50 + unfrozen layer4 (3ep)"
RUN_FOR_CM = preferred if preferred in RUN_DATA else (next(iter(RUN_DATA.keys())) if RUN_DATA else None)

if RUN_FOR_CM is None:
    display(Markdown("**No runs available for confusion matrix view.**"))
else:
    ev = RUN_DATA[RUN_FOR_CM].get("eval")
    if not ev:
        display(Markdown(f"**No eval report available for:** `{RUN_FOR_CM}`"))
    else:
        cm = np.array(ev["metrics"]["confusion"])
        plt.figure(figsize=(4, 4))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["good", "defect"], yticklabels=["good", "defect"])
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.title(f"Confusion Matrix - {RUN_FOR_CM}")
        plt.tight_layout()
        plt.show()
        fp = int(cm[0, 1])
        fn = int(cm[1, 0])
        print(f"False Positives: {fp} | False Negatives: {fn}")


## 5) Failure analysis: misclassifications and false negatives


In [ ]:
preferred = "ResNet50 + unfrozen layer4 (3ep)"
RUN_FOR_FAILURE = preferred if preferred in RUN_DATA else (next(iter(RUN_DATA.keys())) if RUN_DATA else None)

if RUN_FOR_FAILURE is None:
    display(Markdown("**No runs available for failure analysis.**"))
    mis_df = pd.DataFrame()
    fn_df = pd.DataFrame()
else:
    mis_path = RUN_DATA[RUN_FOR_FAILURE]["misclassified_csv"]
    fn_path = RUN_DATA[RUN_FOR_FAILURE]["false_negatives_csv"]

    mis_df = pd.read_csv(mis_path) if mis_path.exists() else pd.DataFrame()
    fn_df = pd.read_csv(fn_path) if fn_path.exists() else pd.DataFrame()

    print("Run:", RUN_FOR_FAILURE)
    print("Misclassified:", len(mis_df))
    print("False negatives:", len(fn_df))

    if not mis_df.empty:
        display(mis_df[["image_path", "target_label", "pred_label", "prob_defect", "prob_good"]].head(20))
    if not fn_df.empty:
        display(Markdown("**False negative samples (critical):**"))
        display(fn_df[["image_path", "prob_defect", "prob_good"]].head(20))


In [ ]:
def _resolve_path(p: str) -> Path:
    pp = Path(p)
    if pp.is_absolute():
        return pp
    return ROOT / pp


def show_image_grid(df: pd.DataFrame, title: str, n: int = 12, cols: int = 4):
    if df.empty:
        print(f"No rows for {title}")
        return

    sample = df.head(n).copy()
    rows = int(np.ceil(len(sample) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.5, rows * 3.2))
    axes = np.array(axes).reshape(-1)
    for ax in axes:
        ax.axis("off")

    for i, (_, r) in enumerate(sample.iterrows()):
        ax = axes[i]
        img_path = _resolve_path(r["image_path"])
        img = cv2.imread(str(img_path))
        if img is None:
            ax.text(0.5, 0.5, f"Missing\n{img_path.name}", ha="center", va="center")
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(
            f"T:{r.get("target_label","?")} P:{r.get("pred_label","?")}\n"
            f"p_def:{r.get("prob_defect", np.nan):.2f} {img_path.name}",
            fontsize=8
        )

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


show_image_grid(fn_df, f"False Negatives - {RUN_FOR_FAILURE}", n=12, cols=4)
show_image_grid(mis_df, f"All Misclassifications - {RUN_FOR_FAILURE}", n=12, cols=4)


## 6) Sample predictions from a saved checkpoint


In [ ]:
from src.eval_baseline import build_model, infer_model_type
from src.transforms import get_eval_transforms

preferred = "ResNet50 + unfrozen layer4 (3ep)"
RUN_FOR_PRED = preferred if preferred in RUN_DATA else (next(iter(RUN_DATA.keys())) if RUN_DATA else None)
N_SAMPLES = 12
IMG_SIZE = 224

if RUN_FOR_PRED is None:
    display(Markdown("**No runs available for prediction visualization.**"))
    pred_df = pd.DataFrame()
else:
    run_dir = RUN_DATA[RUN_FOR_PRED]["run_dir"]
    ckpt_path = run_dir / "best.pt"
    if not ckpt_path.exists():
        display(Markdown(f"**Checkpoint missing:** `{ckpt_path}`"))
        pred_df = pd.DataFrame()
    elif manifest_df.empty:
        display(Markdown("**Manifest is empty; cannot sample prediction images.**"))
        pred_df = pd.DataFrame()
    else:
        model_type, pretrained, unfreeze_last_block = infer_model_type(ckpt_path)
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        # Use pretrained=False here to avoid any external downloads; checkpoint weights are loaded right after.
        model = build_model(model_type, pretrained=False, unfreeze_last_block=unfreeze_last_block).to(device)
        state = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(state)
        model.eval()

        eval_tfm = get_eval_transforms(img_size=IMG_SIZE)
        test_pool = manifest_df[manifest_df["split"] == "test"]
        if test_pool.empty:
            display(Markdown("**No test split rows found in manifest.**"))
            pred_df = pd.DataFrame()
        else:
            test_df = test_pool.sample(n=min(N_SAMPLES, len(test_pool)), random_state=42).reset_index(drop=True)

            pred_rows = []
            for _, row in test_df.iterrows():
                img_path = _resolve_path(str(row["image_path"]))
                img = cv2.imread(str(img_path))
                if img is None:
                    continue
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                tensor = eval_tfm(image=img_rgb)["image"].unsqueeze(0).to(device)
                with torch.no_grad():
                    logits = model(tensor)
                    probs = torch.softmax(logits, dim=1).squeeze(0).cpu().numpy()
                    pred = int(np.argmax(probs))
                pred_rows.append({
                    "image_path": str(img_path),
                    "target": int(row["label"]),
                    "target_label": row["label_str"],
                    "pred": pred,
                    "pred_label": "defect" if pred == 1 else "good",
                    "prob_good": float(probs[0]),
                    "prob_defect": float(probs[1]),
                })

            pred_df = pd.DataFrame(pred_rows)
            if pred_df.empty:
                display(Markdown("**Could not load sample prediction images.**"))
            else:
                display(pred_df)
                show_image_grid(pred_df, f"Sample Predictions - {RUN_FOR_PRED}", n=len(pred_df), cols=4)


## 7) Baseline analysis summary and next decisions


In [ ]:
if 'comparison_df' not in globals() or comparison_df.empty:
    display(Markdown("**No comparison metrics available yet. Run evaluations first.**"))
else:
    best_recall_row = comparison_df.sort_values("recall_defect", ascending=False).iloc[0]
    best_f1_row = comparison_df.sort_values("f1_defect", ascending=False).iloc[0]

    defect_count = int((manifest_df['label_str'] == 'defect').sum()) if not manifest_df.empty else 0
    good_count = int((manifest_df['label_str'] == 'good').sum()) if not manifest_df.empty else 0

    summary_md = f"""
### Baseline Analysis
- Dataset size: **{len(manifest_df)}** images
- Class balance: defect={defect_count}, good={good_count}
- Best recall(defect) run: **{best_recall_row['run']}** (recall_defect={best_recall_row['recall_defect']:.4f})
- Best F1(defect) run: **{best_f1_row['run']}** (f1_defect={best_f1_row['f1_defect']:.4f})

### Data-driven next improvements
1. If FN cases are subtle/small cracks: test `img_size=384` and multi-scale crops.
2. If lighting shifts affect failures: increase illumination augmentations and contrast normalization.
3. If recall drops with longer training: run threshold tuning on validation to prioritize defect recall.
4. Audit persistent FN filenames for potential label ambiguity before architecture changes.
"""

    display(Markdown(summary_md))
